# 02 - Data Cleaning
This notebook removes unwanted/redundant columns, cleans missing age values and encodes categorical variables

### Imports & Data Load

In [4]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import OrdinalEncoder

In [5]:
data = pd.read_csv("../Data/labeled_data.csv")
cols = list(data)
print(cols)
data.head()


['Student ID', 'Age', 'Grade', 'School', 'Q1', 'Q1_Response Time Score', 'Q2', 'Q2_Response Time Score', 'Q3', 'Q3_Response Time Score', 'Q4', 'Q4_Response Time Score', 'Q5', 'Q5_Response Time Score', 'Q6', 'Q6_Response Time Score', 'Q7', 'Q7_Response Time Score', 'Q8', 'Q8_Response Time Score', 'Q9', 'Q9_Response Time Score', 'Q10', 'Q10_Response Time Score', 'Q11', 'Q11_Response Time Score', 'Q12', 'Q12_Response Time Score', 'Q13', 'Q13_Response Time Score', 'Q14', 'Q14_Response Time Score', 'Q15', 'Q15_Response Time Score', 'Q16', 'Q16_Response Time Score', 'Q17', 'Q17_Response Time Score', 'Q18', 'Q18_Response Time Score', 'Q19', 'Q19_Response Time Score', 'Q20', 'Q20_Response Time Score', 'Q21', 'Q21_Response Time Score', 'Q22', 'Q22_Response Time Score', 'Q23', 'Q23_Response Time Score', 'Q24', 'Q24_Response Time Score', 'Q1_Time', 'Q1_Time by Grade', 'Q2_Time', 'Q2_Time by Grade', 'Q3_Time', 'Q3_Time by Grade', 'Q4_Time', 'Q4_Time by Grade', 'Q5_Time', 'Q5_Time by Grade', 'Q6_Ti

,Student ID,Age,Grade,School,Q1,Q1_Response Time Score,Q2,Q2_Response Time Score,Q3,Q3_Response Time Score,...,Do you think your child particularly struggles in mathematics?,Performance in Math\n(original grades),Performance in Math\n(derived grades),Performance in Other Subjects\n(original grades),Performance in Other Subjects\n(derived grades),Do you think the child struggles specifically in Math?,Efficiency_Score,Cluster,Cluster_Probability,Risk_Label
0,_ anaya-3,NaN,3,NEWLANDS,1,0,1,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.9208,5,1.0,Moderate Risk
1,_abdullah hassan-2,NaN,2,NEWLANDS,1,1,0,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.8491,1,1.0,No Risk
2,3-Jan,NaN,3,NEWLANDS,1,0,1,0,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,0.8070,1,1.0,No Risk
3,1016,8.0,2,AL-BAYAN,1,0,1,0,1,0,...,No,Meeting Class Benchmarks,Good,Meeting Class Benchmarks,Good,No,1.1618,1,1.0,No Risk
4,1017,8.0,2,AL-BAYAN,1,0,1,0,1,0,...,NaN,62.50%,Above Average,80.00%,Good,No,1.2760,1,1.0,No Risk


In [11]:
# get actual moderate risk rows and their API-ready values
moderate_samples = data[data['Risk_Label'] == 'Moderate Risk'].sample(10, random_state=42)
print(moderate_samples[['Total Score', 'Grade', 'Age'] + 
                         [col for col in data.columns 
                          if col.startswith('Q')]].to_string())

     Total Score  Grade   Age  Q1  Q1_Response Time Score  Q2  Q2_Response Time Score  Q3  Q3_Response Time Score  Q4  Q4_Response Time Score  Q5  Q5_Response Time Score  Q6  Q6_Response Time Score  Q7  Q7_Response Time Score  Q8  Q8_Response Time Score  Q9  Q9_Response Time Score  Q10  Q10_Response Time Score  Q11  Q11_Response Time Score  Q12  Q12_Response Time Score  Q13  Q13_Response Time Score  Q14  Q14_Response Time Score  Q15  Q15_Response Time Score  Q16  Q16_Response Time Score  Q17  Q17_Response Time Score  Q18  Q18_Response Time Score  Q19  Q19_Response Time Score  Q20  Q20_Response Time Score  Q21  Q21_Response Time Score  Q22  Q22_Response Time Score  Q23  Q23_Response Time Score  Q24  Q24_Response Time Score  Q1_Time    Q1_Time by Grade  Q2_Time    Q2_Time by Grade  Q3_Time    Q3_Time by Grade  Q4_Time    Q4_Time by Grade  Q5_Time    Q5_Time by Grade  Q6_Time    Q6_Time by Grade  Q7_Time    Q7_Time by Grade  Q8_Time    Q8_Time by Grade  Q9_Time    Q9_Time by Grade  Q10_Ti

### Removing unwanted Columns

In [79]:
cols_to_drop = (
    [col for col in data.columns if 'Response Time Score' in col] + 
    [col for col in data.columns if col.endswith('_Time') and 'by Grade' not in col] +  
    ['School'] +
    ['Total Score'] +
    ['Performance in Math\n(original grades)'] +
    ['Performance in Other Subjects\n(original grades)']
)

data = data.drop(columns=cols_to_drop)
cols = list(data)
cols

['Student ID',
 'Age',
 'Grade',
 'Q1',
 'Q2',
 'Q3',
 'Q4',
 'Q5',
 'Q6',
 'Q7',
 'Q8',
 'Q9',
 'Q10',
 'Q11',
 'Q12',
 'Q13',
 'Q14',
 'Q15',
 'Q16',
 'Q17',
 'Q18',
 'Q19',
 'Q20',
 'Q21',
 'Q22',
 'Q23',
 'Q24',
 'Q1_Time by Grade',
 'Q2_Time by Grade',
 'Q3_Time by Grade',
 'Q4_Time by Grade',
 'Q5_Time by Grade',
 'Q6_Time by Grade',
 'Q7_Time by Grade',
 'Q8_Time by Grade',
 'Q9_Time by Grade',
 'Q10_Time by Grade',
 'Q11_Time by Grade',
 'Q12_Time by Grade',
 'Q13_Time by Grade',
 'Q14_Time by Grade',
 'Q15_Time by Grade',
 'Q16_Time by Grade',
 'Q17_Time by Grade',
 'Q18_Time by Grade',
 'Q19_Time by Grade',
 'Q20_Time by Grade',
 'Q21_Time by Grade',
 'Q22_Time by Grade',
 'Q23_Time by Grade',
 'Q24_Time by Grade',
 'How much did the student enjoy the activity',
 'How is the student feeling',
 "Was your child's birth weight considered very low (3 pounds or less)?",
 'Was your child born prematurely (before 37 weeks)?',
 'Any birth complications or known birth defects?',
 'Is 

### Replacing Null Ages with Average Age for the Grade

In [80]:
null_counts = data['Age'].isnull().sum()
print(null_counts[null_counts > 0])

[358]


In [81]:
data['Age'] = data.groupby('Grade')['Age'].transform(
    lambda x: x.fillna(x.mean().round(1))
)

print("Missing ages after imputation:", data['Age'].isnull().sum())
data['Age'].describe()

Missing ages after imputation: 0


count    680.000000
mean       8.377221
std        0.950057
min        5.000000
25%        8.000000
50%        8.400000
75%        8.900000
max       13.000000
Name: Age, dtype: float64

### Encoding Categorical Variables for Response Time by Grade

In [82]:
response_time_mapping = {
    'Way Above Avg': 1,
    'Slightly Above Avg': 2,
    'Average': 3,
    'Slightly Below Avg': 4,
    'Way Below Avg': 5
}

response_time_cols = [
    col for col in data.columns if col.endswith('_Time by Grade')
]

for col in response_time_cols:
    data[col] = data[col].str.strip().str.title()
    data[col] = data[col].map(response_time_mapping)

data[response_time_cols].describe()    

,Q1_Time by Grade,Q2_Time by Grade,Q3_Time by Grade,Q4_Time by Grade,Q5_Time by Grade,Q6_Time by Grade,Q7_Time by Grade,Q8_Time by Grade,Q9_Time by Grade,Q10_Time by Grade,...,Q15_Time by Grade,Q16_Time by Grade,Q17_Time by Grade,Q18_Time by Grade,Q19_Time by Grade,Q20_Time by Grade,Q21_Time by Grade,Q22_Time by Grade,Q23_Time by Grade,Q24_Time by Grade
count,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,...,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000
mean,3.000000,2.998529,2.998529,3.000000,2.998529,3.000000,3.000000,3.000000,2.998529,2.995588,...,3.000000,3.000000,3.000000,3.000000,2.998529,3.000000,2.998529,2.998529,3.000000,3.000000
std,1.105616,1.104948,1.104948,1.100274,1.103615,1.105616,1.105616,1.104283,1.104948,1.102272,...,1.105616,1.105616,1.105616,1.105616,1.103615,1.105616,1.103615,1.104948,1.105616,1.105616
min,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
25%,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
50%,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,...,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000,3.000000
75%,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,...,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000,4.000000
max,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,...,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000,5.000000


### Encoding Categorical Variables for Family & Birth History

In [83]:
family_cols = [
    'Was your child\'s birth weight considered very low (3 pounds or less)?',
    'Was your child born prematurely (before 37 weeks)?',
    'Any birth complications or known birth defects?',
    'Is there a family history (parent/sibling) of significant difficulty with mathematics or diagnosed dyscalculia?',
    'Do you think your child particularly struggles in mathematics?'
]

mapping = {'Yes': 1, 'No': 0}
for col in family_cols:
    data[col] = data[col].map(mapping).fillna(2).astype(int)

for col in family_cols:
    print(data[col].value_counts())
    print()

Was your child's birth weight considered very low (3 pounds or less)?
2    474
0    206
Name: count, dtype: int64

Was your child born prematurely (before 37 weeks)?
2    473
0    196
1     11
Name: count, dtype: int64

Any birth complications or known birth defects?
2    470
0    206
1      4
Name: count, dtype: int64

Is there a family history (parent/sibling) of significant difficulty with mathematics or diagnosed dyscalculia?
2    476
0    178
1     26
Name: count, dtype: int64

Do you think your child particularly struggles in mathematics?
2    470
0    173
1     37
Name: count, dtype: int64



### Encoding Categorical Variables for Academic Performance

In [84]:
performance_mapping = {
    'Below Average': 1,
    'Average': 2,
    'Above Average': 3,
    'Good': 4,
    'Excellent': 5
}

academic_cols = [
    'Performance in Math\n(derived grades)',
    'Performance in Other Subjects\n(derived grades)'
]

for col in academic_cols:
    data[col] = data[col].str.strip().str.title()
    data[col] = data[col].map(performance_mapping).fillna(0).astype(int)

for col in academic_cols:
    print(data[col].value_counts())
    print()
    

Performance in Math\n(derived grades)
0    183
4    161
5    130
3    107
2     71
1     28
Name: count, dtype: int64

Performance in Other Subjects\n(derived grades)
4    194
0    183
3    112
5     90
2     75
1     26
Name: count, dtype: int64



In [85]:
data['Do you think the child struggles specifically in Math?'] = data['Do you think the child struggles specifically in Math?'].map({'Yes': 1, 'No': 0}).fillna(2).astype(int)
print(data['Do you think the child struggles specifically in Math?'].value_counts())

Do you think the child struggles specifically in Math?
0    403
2    183
1     94
Name: count, dtype: int64


### Encoding Target Variable

In [86]:
data['Risk_Label'].head()

0    Moderate Risk
1          No Risk
2          No Risk
3          No Risk
4          No Risk
Name: Risk_Label, dtype: str

In [87]:
label_mapping = {
    'No Risk': 0,
    'Moderate Risk': 1,
    'Severe Risk': 2
}

data['Risk_Label'] = data['Risk_Label'].map(label_mapping)

In [88]:
data['Risk_Label'].value_counts()
data['Risk_Label'].value_counts(normalize=True) * 100

Risk_Label
0    61.764706
1    27.941176
2    10.294118
Name: proportion, dtype: float64

### Combining Individual Question Scores & Response Times by Categories

In [89]:
category_map = {
    'Counting': ['Q1', 'Q2'],
    'Subitising': ['Q3', 'Q4', 'Q24'],
    'Magnitude': ['Q5', 'Q6', 'Q10'],
    'Place Value': ['Q7', 'Q8', 'Q9', 'Q11'],
    'Number Line': ['Q12', 'Q13'],
    'Arithmetic': ['Q14', 'Q15'],
    'Fractions': ['Q16', 'Q17', 'Q18'],
    'Money': ['Q19', 'Q20', 'Q21'],
    'Time': ['Q22', 'Q23']
}

for category, questions in category_map.items():
    data[f'Score_{category}'] = data[questions].mean(axis=1)
    

    time_cols = [f'{q}_Time by Grade' for q in questions 
                 if f'{q}_Time by Grade' in data.columns]
    if time_cols:
        data[f'Time_{category}'] = data[time_cols].mean(axis=1).round(1)

data.drop(columns=[col for col in data.columns if col.startswith('Q')], inplace=True)

In [90]:
data[[col for col in data.columns if col.startswith('Score_') or col.startswith('Time_') ]].describe()

,Score_Counting,Time_Counting,Score_Subitising,Time_Subitising,Score_Magnitude,Time_Magnitude,Score_Place Value,Time_Place Value,Score_Number Line,Time_Number Line,Score_Arithmetic,Time_Arithmetic,Score_Fractions,Time_Fractions,Score_Money,Time_Money,Score_Time,Time_Time
count,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000,680.000000
mean,0.848529,2.999265,0.976961,3.000294,0.809314,2.997206,0.891176,2.999706,0.674265,2.999265,0.761029,2.998529,0.458333,2.999412,0.588235,2.998676,0.873529,2.999265
std,0.275132,0.955759,0.102138,0.848806,0.272707,0.888583,0.187798,0.895711,0.416830,0.924825,0.295816,0.936888,0.293210,0.823531,0.357804,0.863299,0.275733,0.922433
min,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000,0.000000,1.000000
25%,0.500000,2.500000,1.000000,2.300000,0.666667,2.300000,0.750000,2.200000,0.250000,2.500000,0.500000,2.500000,0.333333,2.300000,0.333333,2.300000,1.000000,2.500000
50%,1.000000,3.000000,1.000000,3.000000,1.000000,3.000000,1.000000,3.000000,1.000000,3.000000,1.000000,3.000000,0.333333,3.000000,0.666667,3.000000,1.000000,3.000000
75%,1.000000,3.500000,1.000000,3.700000,1.000000,3.700000,1.000000,3.500000,1.000000,3.500000,1.000000,3.500000,0.666667,3.700000,1.000000,3.700000,1.000000,3.500000
max,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000,1.000000,5.000000


### Determining Performance Gap in Math vs. Other Subjects

In [91]:
math_col = 'Performance in Math\n(derived grades)'
other_col = 'Performance in Other Subjects\n(derived grades)'

data['Performance Gap'] = data[math_col] - data[other_col]

print(data['Performance Gap'].describe())

data.drop(columns=[col for col in data.columns if col.startswith('Performance in')], inplace=True)

count    680.000000
mean       0.069118
std        0.648723
min       -3.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        3.000000
Name: Performance Gap, dtype: float64


In [92]:
renamed_cols = {
    'Was your child\'s birth weight considered very low (3 pounds or less)?': 'Low_Birth_Weight',
    'Was your child born prematurely (before 37 weeks)?': 'Born_Prematurely',
    'Any birth complications or known birth defects?': 'Birth_Complications',   
    'Is there a family history (parent/sibling) of significant difficulty with mathematics or diagnosed dyscalculia?': 'Family_Math_Difficulty',
    'Do you think your child particularly struggles in mathematics?': 'Parent_Perception_Math',
    'Do you think the child struggles specifically in Math?': 'Teacher_Perception_Math',
    'How much did the student enjoy the activity': 'Enjoyment_Score',
    'How is the student feeling': 'Student_Feeling'
    }

data.rename(columns=renamed_cols, inplace=True)
data.columns

Index(['Student ID', 'Age', 'Grade', 'Enjoyment_Score', 'Student_Feeling',
       'Low_Birth_Weight', 'Born_Prematurely', 'Birth_Complications',
       'Family_Math_Difficulty', 'Parent_Perception_Math',
       'Teacher_Perception_Math', 'Efficiency_Score', 'Cluster',
       'Cluster_Probability', 'Risk_Label', 'Score_Counting', 'Time_Counting',
       'Score_Subitising', 'Time_Subitising', 'Score_Magnitude',
       'Time_Magnitude', 'Score_Place Value', 'Time_Place Value',
       'Score_Number Line', 'Time_Number Line', 'Score_Arithmetic',
       'Time_Arithmetic', 'Score_Fractions', 'Time_Fractions', 'Score_Money',
       'Time_Money', 'Score_Time', 'Time_Time', 'Performance Gap'],
      dtype='str')

### Data Validations

In [93]:
data.shape

(680, 34)

In [94]:
print(data.dtypes)

Student ID                     str
Age                        float64
Grade                        int64
Enjoyment_Score              int64
Student_Feeling              int64
Low_Birth_Weight             int64
Born_Prematurely             int64
Birth_Complications          int64
Family_Math_Difficulty       int64
Parent_Perception_Math       int64
Teacher_Perception_Math      int64
Efficiency_Score           float64
Cluster                      int64
Cluster_Probability        float64
Risk_Label                   int64
Score_Counting             float64
Time_Counting              float64
Score_Subitising           float64
Time_Subitising            float64
Score_Magnitude            float64
Time_Magnitude             float64
Score_Place Value          float64
Time_Place Value           float64
Score_Number Line          float64
Time_Number Line           float64
Score_Arithmetic           float64
Time_Arithmetic            float64
Score_Fractions            float64
Time_Fractions      

In [95]:
data.drop(columns=['Cluster', 'Cluster_Probability', 'Efficiency_Score'], inplace=True)

In [96]:
null_counts = data.isnull().sum()
print(null_counts[null_counts > 0])
print('\nTotal NaNs:', data.isnull().sum().sum())

Series([], dtype: int64)

Total NaNs: 0


In [97]:
print(data['Risk_Label'].value_counts())
print(data['Risk_Label'].value_counts(normalize=True) * 100)

Risk_Label
0    420
1    190
2     70
Name: count, dtype: int64
Risk_Label
0    61.764706
1    27.941176
2    10.294118
Name: proportion, dtype: float64


### Saving Cleaned Data

In [98]:
data.to_csv("../Data/cleaned.csv", index=False)